In [2]:
import numpy as np
import pandas as pd
import pysam
import pyranges as pr

In [ ]:
# --- paths -------------------------------------------------------------------
# Defaults are the cluster locations used for the published run. Override via
# environment variables to run elsewhere; SHA-256 for each input is in the README.
import os
from pathlib import Path

DATA_ROOT  = Path(os.environ.get("MULTIOME_DATA_ROOT", "/data1/normantm/eli"))
PERTURBSEQ = Path(os.environ.get("PERTURBSEQ_PATH", DATA_ROOT / "software"))
RESULTS    = Path(os.environ.get("MULTIOME_RESULTS", Path.cwd().parent / "results"))

SHARE      = DATA_ROOT / "T7" / "202404_SIRLOIN_multiome" / "share"
GENOME_DIR = DATA_ROOT / "seq2gex" / "data" / "genome"

ATAC_H5AD    = SHARE / "atac_singlets_macs3_peaks.h5ad"
GEX_HDF5     = SHARE / "gex_norm_regressed.hdf5"
PROMOTER_BED = GENOME_DIR / "epdNewHuman006_extended_promoter_regions.bed"
GENES_GTF    = GENOME_DIR / "genes.gtf"
GENOME_FA    = GENOME_DIR / "genome.fa"

GUIDES_CSV        = Path("multiome_paper_igvf_guides.csv")
GEX_EFFECTS_CSV   = RESULTS / "multiome_paper_guide_effect_matrix.csv"
ATAC_PEAKS_CSV    = RESULTS / "multiome_paper_differential_peaks_by_guide.csv"


In [3]:
df = pd.read_csv(GUIDES_CSV, index_col = 0)
df

,guide_id,spacer,targeting,type,guide_chr,guide_start,guide_end,strand,pam,genomic_element,...,n_offtargets,n_tss_options,is_upstream,in_promoter,tss_start,tss_end,tss_strand,tss_distance,pick_order,library
0,SMARCE1_GAGCGACCTCAGGAAGCCGT,GAGCGACCTCAGGAAGCCGT,True,targeting,chr17,40647463.0,40647483.0,-,CCN,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SMARCB1_GGCCTGGTCGTCGTCTGCGG,GGCCTGGTCGTCGTCTGCGG,True,targeting,chr22,23786988.0,23787008.0,+,NGG,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARID1A_GTTTCTCCGGCAGCAGAAAG,GTTTCTCCGGCAGCAGAAAG,True,targeting,chr1,26696017.0,26696037.0,+,NGG,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMARCA4_GGGAGGCGCCGGGAAGTCGA,GGGAGGCGCCGGGAAGTCGA,True,targeting,chr19,10961136.0,10961156.0,+,NGG,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DPF2_GCTGCGCGCTGCGGACTGTG,GCTGCGCGCTGCGGACTGTG,True,targeting,chr11,65333832.0,65333852.0,+,NGG,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SMARCC2_GCCCGAGCCGGAGAAGATGG,GCCCGAGCCGGAGAAGATGG,True,targeting,chr12,56189457.0,56189477.0,-,CCN,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SMARCC1_GGGTGCGCGCGGGAACGACC,GGGTGCGCGCGGGAACGACC,True,targeting,chr3,47781853.0,47781873.0,-,CCN,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,EP400_GCGCTCCGGCCGCGTCAGGA,GCGCTCCGGCCGCGTCAGGA,True,targeting,chr12,131950000.0,131950020.0,-,CCN,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ACTL6A_GCCCACCTCAGGCAACAAAG,GCCCACCTCAGGCAACAAAG,True,targeting,chr3,179562959.0,179562979.0,-,CCN,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DMAP1_GCAGCTGCCGGACCCAGGTG,GCAGCTGCCGGACCCAGGTG,True,targeting,chr1,44213510.0,44213530.0,+,NGG,promoter,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
def reverse_complement(seq):
    return seq.translate(str.maketrans("ACGT", "TGCA"))[::-1]

def find_matches(seq, query, max_mismatches=1):
    """
    Return all positions where query matches seq with <= max_mismatches.
    """
    L = len(query)
    hits = []

    for i in range(len(seq) - L + 1):
        window = seq[i:i + L]
        mismatches = sum(a != b for a, b in zip(window, query))
        if mismatches <= max_mismatches:
            hits.append(i)

    return hits


def fill_guide_coordinates(
    df,
    fasta_path = GENOME_FA,
    gtf_path = GENES_GTF,
    promoter_padding=5000,
):
    """
    Fill guide_chr, guide_start, guide_end, strand by searching for each
    spacer sequence within the target gene locus.

    Parameters
    ----------
    df : pandas.DataFrame
    fasta_path : str
        Reference genome FASTA (must be indexed with samtools faidx).
    gtf_path : str
        Gene annotation (GTF).
    promoter_padding : int
        Extra bp to search upstream/downstream of the gene.

    Returns
    -------
    pandas.DataFrame
    """

    genome = pysam.FastaFile(str(fasta_path))

    genes = (
        pr.read_gtf(str(gtf_path))
        .df.query("Feature == 'gene'")
        [["Chromosome", "Start", "End", "Strand", "gene_name"]]
        .drop_duplicates("gene_name")
        .set_index("gene_name")
    )

    for i, row in df.iterrows():

        if pd.isna(row.gene_symbol):
            continue

        if row.gene_symbol not in genes.index:
            continue

        gene = genes.loc[row.gene_symbol]

        chrom = gene.Chromosome
        start = max(0, gene.Start - promoter_padding)
        end = gene.End + promoter_padding

        seq = genome.fetch(chrom, start, end).upper()

        spacer = row.spacer.upper()
        rc = reverse_complement(spacer)

        hits = []

        for pos in find_matches(seq, spacer, max_mismatches=1):
            hits.append((chrom, start + pos, "+"))

        for pos in find_matches(seq, rc, max_mismatches=1):
            hits.append((chrom, start + pos, "-"))

        if len(hits) == 1:
            chrom, guide_start, strand = hits[0]

            df.at[i, "guide_chr"] = chrom
            df.at[i, "guide_start"] = guide_start
            df.at[i, "guide_end"] = guide_start + len(spacer)
            df.at[i, "strand"] = strand

        elif len(hits) == 0:
            print(f"No hit for {row.guide_id}")

        else:
            print(f"Multiple hits for {row.guide_id}: {len(hits)}")

    return df

In [5]:
df_guides = fill_guide_coordinates(df)

In [6]:
df_guides['pam'] = df_guides.strand.map({"+": 'NGG', "-": "CCN"})
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_guides)

,guide_id,spacer,targeting,type,guide_chr,guide_start,guide_end,strand,pam,genomic_element,intended_target_name,intended_target_chr,intended_target_start,intended_target_end,putative_target_genes,reporter,imperfect,gene_symbol,position_category,n_offtargets,n_tss_options,is_upstream,in_promoter,tss_start,tss_end,tss_strand,tss_distance,pick_order,library
0,SMARCE1_GAGCGACCTCAGGAAGCCGT,GAGCGACCTCAGGAAGCCGT,True,targeting,chr17,40647463.0,40647483.0,-,CCN,promoter,ENSG00000073584,chr17,40647806.0,40648816.0,NaN,NaN,NaN,SMARCE1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SMARCB1_GGCCTGGTCGTCGTCTGCGG,GGCCTGGTCGTCGTCTGCGG,True,targeting,chr22,23786988.0,23787008.0,+,NGG,promoter,ENSG00000099956,chr22,23785978.0,23786988.0,NaN,NaN,NaN,SMARCB1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARID1A_GTTTCTCCGGCAGCAGAAAG,GTTTCTCCGGCAGCAGAAAG,True,targeting,chr1,26696017.0,26696037.0,+,NGG,promoter,ENSG00000117713,chr1,26695032.0,26696042.0,NaN,NaN,NaN,ARID1A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMARCA4_GGGAGGCGCCGGGAAGTCGA,GGGAGGCGCCGGGAAGTCGA,True,targeting,chr19,10961136.0,10961156.0,+,NGG,promoter,ENSG00000127616,chr19,10960030.0,10961040.0,NaN,NaN,NaN,SMARCA4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DPF2_GCTGCGCGCTGCGGACTGTG,GCTGCGCGCTGCGGACTGTG,True,targeting,chr11,65333832.0,65333852.0,+,NGG,promoter,ENSG00000133884,chr11,65332867.0,65333877.0,NaN,NaN,NaN,DPF2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SMARCC2_GCCCGAGCCGGAGAAGATGG,GCCCGAGCCGGAGAAGATGG,True,targeting,chr12,56189457.0,56189477.0,-,CCN,promoter,ENSG00000139613,chr12,56189462.0,56190472.0,NaN,NaN,NaN,SMARCC2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SMARCC1_GGGTGCGCGCGGGAACGACC,GGGTGCGCGCGGGAACGACC,True,targeting,chr3,47781853.0,47781873.0,-,CCN,promoter,ENSG00000173473,chr3,47781882.0,47782892.0,NaN,NaN,NaN,SMARCC1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,EP400_GCGCTCCGGCCGCGTCAGGA,GCGCTCCGGCCGCGTCAGGA,True,targeting,chr12,131950000.0,131950020.0,-,CCN,promoter,ENSG00000183495,chr12,131948964.0,131949974.0,NaN,NaN,NaN,EP400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ACTL6A_GCCCACCTCAGGCAACAAAG,GCCCACCTCAGGCAACAAAG,True,targeting,chr3,179562959.0,179562979.0,-,CCN,promoter,ENSG00000136518,chr3,179561926.0,179562936.0,NaN,NaN,NaN,ACTL6A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DMAP1_GCAGCTGCCGGACCCAGGTG,GCAGCTGCCGGACCCAGGTG,True,targeting,chr1,44213510.0,44213530.0,+,NGG,promoter,ENSG00000178028,chr1,44213867.0,44214877.0,NaN,NaN,NaN,DMAP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
promoters = pd.read_csv(PROMOTER_BED, sep = '\t', header = None)
promoters.columns = ['chr', 'start', 'end', 'gene', 'score', 'strand', 'tss_start', 'tss_end']
promoters['gene'] = promoters.gene.astype(str)

In [8]:
df_guides['intended_target_chr'] = df_guides['guide_chr']
def get_promoter_coords(df):
    if pd.isna(df.gene_symbol):
        return np.nan, np.nan
    else:
        gene = df.gene_symbol
        prom = promoters[promoters["gene"].str.contains(df.gene_symbol, na=False)]
        prom['guide_dist'] = np.abs(prom.start - df.guide_start)
        prom = prom.sort_values('guide_dist').head(1)
        return prom.start.item(), prom.end.item()

df_guides[['intended_target_start', 'intended_target_end']] = df_guides.apply(get_promoter_coords, axis = 1, result_type = 'expand')

In [9]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_guides)

,guide_id,spacer,targeting,type,guide_chr,guide_start,guide_end,strand,pam,genomic_element,intended_target_name,intended_target_chr,intended_target_start,intended_target_end,putative_target_genes,reporter,imperfect,gene_symbol,position_category,n_offtargets,n_tss_options,is_upstream,in_promoter,tss_start,tss_end,tss_strand,tss_distance,pick_order,library
0,SMARCE1_GAGCGACCTCAGGAAGCCGT,GAGCGACCTCAGGAAGCCGT,True,targeting,chr17,40647463.0,40647483.0,-,CCN,promoter,ENSG00000073584,chr17,40647806.0,40648816.0,NaN,NaN,NaN,SMARCE1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,SMARCB1_GGCCTGGTCGTCGTCTGCGG,GGCCTGGTCGTCGTCTGCGG,True,targeting,chr22,23786988.0,23787008.0,+,NGG,promoter,ENSG00000099956,chr22,23785978.0,23786988.0,NaN,NaN,NaN,SMARCB1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ARID1A_GTTTCTCCGGCAGCAGAAAG,GTTTCTCCGGCAGCAGAAAG,True,targeting,chr1,26696017.0,26696037.0,+,NGG,promoter,ENSG00000117713,chr1,26695032.0,26696042.0,NaN,NaN,NaN,ARID1A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SMARCA4_GGGAGGCGCCGGGAAGTCGA,GGGAGGCGCCGGGAAGTCGA,True,targeting,chr19,10961136.0,10961156.0,+,NGG,promoter,ENSG00000127616,chr19,10960030.0,10961040.0,NaN,NaN,NaN,SMARCA4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,DPF2_GCTGCGCGCTGCGGACTGTG,GCTGCGCGCTGCGGACTGTG,True,targeting,chr11,65333832.0,65333852.0,+,NGG,promoter,ENSG00000133884,chr11,65332867.0,65333877.0,NaN,NaN,NaN,DPF2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,SMARCC2_GCCCGAGCCGGAGAAGATGG,GCCCGAGCCGGAGAAGATGG,True,targeting,chr12,56189457.0,56189477.0,-,CCN,promoter,ENSG00000139613,chr12,56189462.0,56190472.0,NaN,NaN,NaN,SMARCC2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,SMARCC1_GGGTGCGCGCGGGAACGACC,GGGTGCGCGCGGGAACGACC,True,targeting,chr3,47781853.0,47781873.0,-,CCN,promoter,ENSG00000173473,chr3,47781882.0,47782892.0,NaN,NaN,NaN,SMARCC1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,EP400_GCGCTCCGGCCGCGTCAGGA,GCGCTCCGGCCGCGTCAGGA,True,targeting,chr12,131950000.0,131950020.0,-,CCN,promoter,ENSG00000183495,chr12,131948964.0,131949974.0,NaN,NaN,NaN,EP400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ACTL6A_GCCCACCTCAGGCAACAAAG,GCCCACCTCAGGCAACAAAG,True,targeting,chr3,179562959.0,179562979.0,-,CCN,promoter,ENSG00000136518,chr3,179561926.0,179562936.0,NaN,NaN,NaN,ACTL6A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,DMAP1_GCAGCTGCCGGACCCAGGTG,GCAGCTGCCGGACCCAGGTG,True,targeting,chr1,44213510.0,44213530.0,+,NGG,promoter,ENSG00000178028,chr1,44213867.0,44214877.0,NaN,NaN,NaN,DMAP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df_guides.to_csv(GUIDES_CSV)

In [11]:
df_effects = df_guides[['guide_id', 'gene_symbol', 'intended_target_name', 'intended_target_chr', 'intended_target_start', 'intended_target_end']]
df_effects

,guide_id,gene_symbol,intended_target_name,intended_target_chr,intended_target_start,intended_target_end
0,SMARCE1_GAGCGACCTCAGGAAGCCGT,SMARCE1,ENSG00000073584,chr17,40647806.0,40648816.0
1,SMARCB1_GGCCTGGTCGTCGTCTGCGG,SMARCB1,ENSG00000099956,chr22,23785978.0,23786988.0
2,ARID1A_GTTTCTCCGGCAGCAGAAAG,ARID1A,ENSG00000117713,chr1,26695032.0,26696042.0
3,SMARCA4_GGGAGGCGCCGGGAAGTCGA,SMARCA4,ENSG00000127616,chr19,10960030.0,10961040.0
4,DPF2_GCTGCGCGCTGCGGACTGTG,DPF2,ENSG00000133884,chr11,65332867.0,65333877.0
5,SMARCC2_GCCCGAGCCGGAGAAGATGG,SMARCC2,ENSG00000139613,chr12,56189462.0,56190472.0
6,SMARCC1_GGGTGCGCGCGGGAACGACC,SMARCC1,ENSG00000173473,chr3,47781882.0,47782892.0
7,EP400_GCGCTCCGGCCGCGTCAGGA,EP400,ENSG00000183495,chr12,131948964.0,131949974.0
8,ACTL6A_GCCCACCTCAGGCAACAAAG,ACTL6A,ENSG00000136518,chr3,179561926.0,179562936.0
9,DMAP1_GCAGCTGCCGGACCCAGGTG,DMAP1,ENSG00000178028,chr1,44213867.0,44214877.0


In [ ]:
import sys
sys.path.append(str(PERTURBSEQ))
from perturbseq import *
import warnings

pop = CellPopulation.from_hdf(str(GEX_HDF5))
pop.genes['gene_name'] = pop.genes.index
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    meanpop = pop.average("guide_identity")

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    ks, ps, qs = ks_de(
        pop, key = "guide_identity", 
        control_cells = "guide_identity == 'NTC'", 
        n_jobs = 24,
)

dfp = ps.drop("NTC", axis = 1).reset_index().rename({'index':'gene'}, axis = 1).melt(id_vars = 'gene', var_name = "guide_target", value_name = "p")
dfq = qs.drop("NTC", axis = 1).reset_index().rename({'index':'gene'}, axis = 1).melt(id_vars = 'gene', var_name = "guide_target", value_name = "q")
dfz = meanpop.matrix.T.reset_index().rename({'index':'gene'}, axis = 1).melt(id_vars = 'gene', var_name = "guide_target", value_name = "zscore")

dfp['id'] = dfp.apply(lambda df: df.gene + "_" + df.guide_target, axis = 1)
dfp = dfp.set_index("id")
dfq['id'] = dfq.apply(lambda df: df.gene + "_" + df.guide_target, axis = 1)
dfq = dfq.set_index("id")
dfz['id'] = dfz.apply(lambda df: df.gene + "_" + df.guide_target, axis = 1)
dfz = dfz.set_index("id")
df_degs = dfz.join(dfp.p).join(dfq.q)
df_degs = df_degs.query("q < 0.1")

In [30]:
df_igvf = df_degs.reset_index(drop = True).rename({"gene": "target_gene", "zscore":"effect_score", "p":"p_val", "q": "p_val_adj", "guide_target": "gene_symbol"}, axis = 1)
df_igvf

,target_gene,gene_symbol,effect_score,p_val,p_val_adj
0,HES4,ACTL6A,-0.022754,3.919103e-05,0.004773
1,AGRN,ACTL6A,-0.093767,8.447696e-07,0.000486
2,C1orf159,ACTL6A,-0.001849,6.355432e-05,0.006577
3,ACAP3,ACTL6A,-0.064005,6.270349e-07,0.000415
4,MXRA8,ACTL6A,-0.355484,1.102844e-07,0.000166
...,...,...,...,...,...
14863,MT2A,YY1,0.150442,2.164686e-07,0.002646
14864,NEDD4L,YY1,-0.298753,5.381424e-09,0.000175
14865,SLX4IP,YY1,-0.164383,1.360412e-05,0.083146
14866,DONSON,YY1,-0.105725,1.335973e-05,0.083146


In [42]:
df_final = df_igvf.merge(df_effects, on = 'gene_symbol')[["effect_score", "p_val",	"p_val_adj", "guide_id", "target_gene",	"intended_target_name",	"intended_target_chr",	"intended_target_start", "intended_target_end"]]
gtf = pr.read_gtf(str(GENES_GTF)).df.query("Feature == 'gene'")
ensembl_dict = dict(zip(gtf.gene_name, gtf.gene_id))
df_final['target_gene'] = df_final.target_gene.map(ensembl_dict)
df_final.to_csv(GEX_EFFECTS_CSV)

In [4]:
import snapatac2 as snap
from scipy.stats import mannwhitneyu, false_discovery_control
from tqdm import tqdm
snap.__version__

'2.6.0'

In [15]:
import snapatac2 as snap
from scipy.stats import mannwhitneyu, false_discovery_control
from tqdm import tqdm

### needs to be run with snapatac2 version 2.6.0, otherwise gives different results than those published in Table S3

atac = snap.read(str(ATAC_H5AD), backed = None)
peaks = snap.tl.merge_peaks(atac.uns['macs3'], snap.genome.hg38)
mtx = snap.pp.make_peak_matrix(atac, use_rep=peaks['Peaks'], counting_strategy='paired-insertion')

guides = mtx.obs.guide_identity.unique().tolist()
guides.remove("NTC")
dfs = []
for guide_target in tqdm(guides):

    selected_peaks = np.logical_or(peaks[guide_target].to_numpy(), peaks['NTC'].to_numpy())
    peak_names = peaks.to_pandas().query(f"{guide_target} or NTC").Peaks

    target_mtx = mtx[mtx.obs.guide_identity == guide_target, selected_peaks].X.toarray()
    ctrl_mtx = mtx[mtx.obs.guide_identity == "NTC", selected_peaks].X.toarray()

    ps = mannwhitneyu(target_mtx, ctrl_mtx, axis = 0).pvalue
    lfcs = np.log2(np.divide(1e-3 + target_mtx.mean(axis=0), 1e-3 + ctrl_mtx.mean(axis = 0)))
    adj_ps = false_discovery_control(ps)

    df_peaks = pd.DataFrame({'feature': peak_names, 'l2fc': lfcs, 'p': ps, 'q': adj_ps, 'guide_target': guide_target})
    dfs.append(df_peaks)

df_mwu = pd.concat(dfs, axis = 0)

100%|██████████| 13/13 [02:17<00:00, 10.54s/it]


In [14]:
mtx

AnnData object with n_obs × n_vars = 4724 × 195831
    obs: 'n_fragment', 'frac_dup', 'frac_mito', 'tsse', 'guide_identity'

In [16]:
df_mwu

,feature,l2fc,p,q,guide_target
0,chr1:9935-10436,-0.183612,0.773328,0.961509,YY1
1,chr1:180634-181135,-0.991516,0.212568,0.794849,YY1
2,chr1:181222-181723,-0.188073,0.723277,0.954271,YY1
4,chr1:629698-630199,0.254574,0.420066,0.879931,YY1
5,chr1:633776-634277,0.154287,0.363081,0.863669,YY1
...,...,...,...,...,...
195821,chrY:11312218-11312719,-1.727947,0.149153,0.671357,SMARCA4
195823,chrY:11314120-11314621,1.160390,0.078878,0.634025,SMARCA4
195825,chrY:11333733-11334234,0.112388,0.952966,0.989665,SMARCA4
195826,chrY:11332969-11333470,-0.452496,0.535625,0.869717,SMARCA4


In [46]:
df_peaks = df_mwu.query("q < 0.1").rename(columns={'l2fc':'effect_score','p': 'p_val', 'q': 'p_val_adj'})
df_peaks['chr'] = df_peaks.feature.str.split(':').str[0]
df_peaks['start'] = df_peaks.feature.str.split(':').str[1].str.split('-').str[0]
df_peaks['end'] = df_peaks.feature.str.split(':').str[1].str.split('-').str[1]

df_guides = pd.read_csv(GEX_EFFECTS_CSV, index_col = 0)[['guide_id', 'intended_target_chr', 'intended_target_start', 'intended_target_end']].drop_duplicates()
df_guides['guide_target'] = df_guides.guide_id.str.split('_').str[0]
df_peaks = df_peaks.merge(df_guides[['guide_target', 'guide_id', 'intended_target_chr', 'intended_target_start', 'intended_target_end']], on='guide_target',how='left')
df_peaks.drop(['feature', 'guide_target'], axis = 1).to_csv(ATAC_PEAKS_CSV)

In [47]:
df_peaks.guide_id.value_counts().sort_index()

guide_id
ACTL6A_GCCCACCTCAGGCAACAAAG     1414
ARID1A_GTTTCTCCGGCAGCAGAAAG     1840
DMAP1_GCAGCTGCCGGACCCAGGTG       206
DPF2_GCTGCGCGCTGCGGACTGTG        124
EP400_GCGCTCCGGCCGCGTCAGGA       716
EZH2_GGTCGCGTCCGACACCCGGT         69
SMARCA4_GGGAGGCGCCGGGAAGTCGA     170
SMARCB1_GGCCTGGTCGTCGTCTGCGG     358
SMARCC1_GGGTGCGCGCGGGAACGACC      73
SMARCC2_GCCCGAGCCGGAGAAGATGG     782
SMARCE1_GAGCGACCTCAGGAAGCCGT    3897
SUZ12_GAGGCTCCGGCGGACCGAGG      1262
YY1_GGCCGGGCCCGAGCAGAGTG          24
Name: count, dtype: int64